# ATS Customer Churn Prediction – Predictive Modelling

This notebook builds predictive classification models to predict customer churn (`Churn`).

**Models:** Logistic Regression and Random Forest  
**Evaluation:** Accuracy, Precision, Recall, F1-score, ROC-AUC, Confusion Matrix and ROC Curve

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)


## 2. Load the ATS dataset

The code first looks for the CSV in the GitHub project `data` folder. If it is not found, it tries the original public dataset URL.

In [ ]:
import os

LOCAL_FILE = 'data/Dataset_ATS_v2_preprocessed.csv'
RAW_GITHUB_FILE = 'https://raw.githubusercontent.com/NithinVarghese97/Customer_Churn_Analysis/main/Data_Preparation/Preprocessed_Dataset/Dataset_ATS_v2_preprocessed.csv'

if os.path.exists(LOCAL_FILE):
    df = pd.read_csv(LOCAL_FILE)
    print('Loaded local GitHub project dataset.')
else:
    df = pd.read_csv(RAW_GITHUB_FILE)
    print('Loaded dataset from the public GitHub URL.')

print('Dataset shape:', df.shape)
display(df.head())


## 3. Inspect the dataset

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())


## 4. Identify and prepare the target variable

The target variable is `Churn`. The code handles common representations such as `Yes/No`, `True/False`, and `1/0`.

In [ ]:
if 'Churn' not in df.columns:
    raise ValueError(
        "The dataset does not contain a 'Churn' column. Available columns are: "
        + str(df.columns.tolist())
    )

# Remove completely empty rows
df = df.dropna(how='all').copy()

# Clean target values
target = df['Churn'].astype(str).str.strip().str.lower()

mapping = {
    'yes': 1, 'y': 1, 'true': 1, '1': 1, 'churned': 1,
    'no': 0, 'n': 0, 'false': 0, '0': 0, 'not churned': 0
}

mapped_target = target.map(mapping)

# If already numeric 0/1, retain it
if mapped_target.notna().all():
    y = mapped_target.astype(int)
else:
    numeric_target = pd.to_numeric(df['Churn'], errors='coerce')
    if numeric_target.notna().all() and set(numeric_target.unique()).issubset({0, 1}):
        y = numeric_target.astype(int)
    else:
        raise ValueError(
            'Unable to convert Churn into a binary 0/1 target. Unique values found: '
            + str(df['Churn'].dropna().unique())
        )

X = df.drop(columns=['Churn']).copy()

# Remove obvious ID columns because they do not provide useful predictive information.
id_like_columns = []
for col in X.columns:
    name = str(col).strip().lower()
    if name in {'id', 'customerid', 'customer_id', 'customer id'} or name.endswith('_id'):
        id_like_columns.append(col)

if id_like_columns:
    X = X.drop(columns=id_like_columns)
    print('Removed ID-like columns:', id_like_columns)

print('Target distribution:')
print(y.value_counts())
print('\nTarget proportions:')
print(y.value_counts(normalize=True).round(3))


## 5. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))


## 6. Automatically detect numeric and categorical features

This makes the notebook robust when the dataset contains a mixture of numbers and text categories.

In [ ]:
numeric_features = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()

print('Numeric features:', numeric_features)
print('\nCategorical features:', categorical_features)


## 7. Preprocessing pipeline

Numeric variables are median-imputed and standardised. Categorical variables are most-frequent imputed and one-hot encoded.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop'
)


## 8. Logistic Regression model

In [ ]:
logistic_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print('Logistic Regression trained successfully.')


## 9. Random Forest model

In [ ]:
random_forest_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)
rf_pred = random_forest_model.predict(X_test)
rf_prob = random_forest_model.predict_proba(X_test)[:, 1]

print('Random Forest trained successfully.')


## 10. Compare model performance

In [ ]:
def model_metrics(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    model_metrics('Logistic Regression', y_test, logistic_pred, logistic_prob),
    model_metrics('Random Forest', y_test, rf_pred, rf_prob)
])

results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
display(results.round(4))


## 11. Classification reports

In [ ]:
print('LOGISTIC REGRESSION')
print(classification_report(y_test, logistic_pred, target_names=['No Churn', 'Churn'], zero_division=0))

print('RANDOM FOREST')
print(classification_report(y_test, rf_pred, target_names=['No Churn', 'Churn'], zero_division=0))


## 12. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, logistic_pred,
    display_labels=['No Churn', 'Churn'],
    ax=axes[0]
)
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred,
    display_labels=['No Churn', 'Churn'],
    ax=axes[1]
)
axes[1].set_title('Random Forest')

plt.tight_layout()
plt.show()


## 13. ROC curve comparison

In [ ]:
logistic_fpr, logistic_tpr, _ = roc_curve(y_test, logistic_prob)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

plt.figure(figsize=(8, 6))
plt.plot(logistic_fpr, logistic_tpr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, logistic_prob):.3f})')
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_prob):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Customer Churn Prediction')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 14. Random Forest feature importance

In [ ]:
rf_preprocessor = random_forest_model.named_steps['preprocessing']
rf_model = random_forest_model.named_steps['model']

feature_names = rf_preprocessor.get_feature_names_out()
importance = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

display(feature_importance.head(20))

plt.figure(figsize=(10, 7))
top_features = feature_importance.head(15).sort_values('Importance')
plt.barh(top_features['Feature'], top_features['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 15 Random Forest Features')
plt.tight_layout()
plt.show()


## 15. Select the best predictive model


In [ ]:
best_model_name = results.iloc[0]['Model']

if best_model_name == 'Logistic Regression':
    best_model = logistic_model
    best_prob = logistic_prob
    best_pred = logistic_pred
else:
    best_model = random_forest_model
    best_prob = rf_prob
    best_pred = rf_pred

print('Best model based on ROC-AUC:', best_model_name)
print('Best model ROC-AUC:', round(results.iloc[0]['ROC-AUC'], 4))


## 16. Generate customer churn risk predictions

Risk bands are based on predicted churn probability:
- Low: below 0.30
- Medium: 0.30 to below 0.60
- High: 0.60 or above

In [ ]:
predictions = X_test.copy()
predictions['Actual_Churn'] = y_test.values
predictions['Predicted_Churn'] = best_pred
predictions['Churn_Probability'] = best_prob

predictions['Risk_Level'] = pd.cut(
    predictions['Churn_Probability'],
    bins=[-np.inf, 0.30, 0.60, np.inf],
    labels=['Low', 'Medium', 'High']
)

display(predictions.head(10))


## 17. Export predictions


In [ ]:
output_file = 'ATS_customer_churn_predictions.csv'
predictions.to_csv(output_file, index=False)
print(f'Predictions saved as: {output_file}')


## 18. Final predictive-modelling conclusion

The project predicts customer churn using two classification algorithms: Logistic Regression and Random Forest. Both models are evaluated using Accuracy, Precision, Recall, F1-score and ROC-AUC. The model with the highest ROC-AUC on the test set is selected as the best predictive model.

The resulting churn probabilities can be used to identify customers with low, medium and high predicted churn risk. These predictions can support retention activities by helping the business prioritise customers who are most likely to churn.

**Important:** This is a predictive model, not a causal analysis. Feature importance indicates predictive usefulness and should not automatically be interpreted as a cause of churn.